# Grounded Paraphraser Qwen3-0.6B Smoke Test

Bu notebook Google Colab icin hazirlanmis grounded paraphraser smoke-test sablonudur.

## A. Title And Warning

- Bu calisma ham waiter assistant egitimi degildir.
- Amac, canonical deterministic cevabi daha dogal ama guvenli Turkce ile paraphrase etmeyi denemektir.
- Model menu urunu, fiyat, alerji iddiasi veya siparis gercegi uydurmamalidir.
- Deterministic grounding katmani dogrulugun ana kaynagidir.
- Bu notebook production-ready degildir.
- Bu repo gorevi icinde hicbir model indirilmedi ve hicbir egitim calistirilmadi.


## B. Runtime Check

Google Colab icinde GPU runtime secin. GPU tipi ve VRAM her oturumda degisebilir.


In [ ]:
import platform
import subprocess
import sys

print('Python:', sys.version)
print('Platform:', platform.platform())

try:
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True, check=False)
    print(result.stdout if result.stdout else result.stderr)
    if result.returncode != 0:
        print('UYARI: GPU gorunmuyor. Colab Runtime -> Change runtime type -> GPU secin.')
except FileNotFoundError:
    print('UYARI: nvidia-smi bulunamadi. GPU runtime aktif olmayabilir.')


## C. Project Setup

Asagidaki iki yoldan yalnizca birini secin.

### Secenek 1: Proje zip dosyasini Colab icine yukle


In [ ]:
from pathlib import Path
from zipfile import ZipFile

PROJECT_ROOT = Path('/content/Garson-bot')
ZIP_PATH = Path('/content/Garson-bot.zip')

# from google.colab import files
# uploaded = files.upload()  # Garson-bot.zip yukleyin
# ZIP_PATH = Path('/content') / next(iter(uploaded))

# with ZipFile(ZIP_PATH, 'r') as archive:
#     archive.extractall('/content')

print('Secenek 1 icin PROJECT_ROOT:', PROJECT_ROOT)


### Secenek 2: Google Drive mount et


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# PROJECT_ROOT = Path('/content/drive/MyDrive/Garson-bot')

# print('Drive PROJECT_ROOT:', PROJECT_ROOT)


## D. Dependency Install

Bu bagimliliklari sadece Colab icinde kurun. Lokal requirements.txt dosyasina eklemeyin.


In [ ]:
%pip install -q transformers datasets peft trl accelerate bitsandbytes sentencepiece


## E. Dataset Path Configuration

Grounded paraphraser train/valid JSONL dosyalarini ve held-out helper dosyalarini kontrol edin.


In [ ]:
import json
from pathlib import Path

TRAIN_FILE = PROJECT_ROOT / 'robot_waiter_ai/datasets/processed/grounded_paraphrase_train.jsonl'
VALID_FILE = PROJECT_ROOT / 'robot_waiter_ai/datasets/processed/grounded_paraphrase_valid.jsonl'
VALID_REFERENCE_FILE = PROJECT_ROOT / 'robot_waiter_ai/evals/grounded_paraphrase_valid_reference.jsonl'
VALID_TEMPLATE_FILE = PROJECT_ROOT / 'robot_waiter_ai/evals/grounded_paraphrase_valid_output_template.jsonl'

for path in [TRAIN_FILE, VALID_FILE, VALID_REFERENCE_FILE, VALID_TEMPLATE_FILE]:
    print(path, '->', path.exists())
    assert path.exists(), f'Eksik dosya: {path}'

def load_jsonl(path: Path):
    with path.open('r', encoding='utf-8') as handle:
        return [json.loads(line) for line in handle if line.strip()]

train_records = load_jsonl(TRAIN_FILE)
valid_records = load_jsonl(VALID_FILE)

print('Train count:', len(train_records))
print('Valid count:', len(valid_records))
assert len(train_records) == 215, 'Beklenen train sayisi 215 olmali'
assert len(valid_records) == 37, 'Beklenen valid sayisi 37 olmali'

for example in train_records[:2]:
    print(json.dumps(example, ensure_ascii=False, indent=2)[:1200])
    print('-' * 80)


## F. Model Configuration

Birincil smoke-test model `Qwen/Qwen3-0.6B` olacaktir. `Qwen/Qwen3-1.7B` yalnizca gerekirse fallback olarak dusunulmelidir.


In [ ]:
MODEL_NAME = 'Qwen/Qwen3-0.6B'
# Fallback adayi: MODEL_NAME = 'Qwen/Qwen3-1.7B'

MAX_SEQ_LENGTH = 1024
OUTPUT_DIR = '/content/grounded_paraphraser_qwen3_0_6b_adapter'
GENERATED_OUTPUTS_PATH = '/content/generated_grounded_paraphrase_qwen3_0_6b_smoke.jsonl'

print('Model:', MODEL_NAME)
print('Bu model smoke-test icindir; final production model degildir.')


## G. Formatting Function

Bu notebook built JSONL `messages` alanini dogrudan kullanir. Prompt gorevi acik olmali: gerekli terimleri koru, yasak terimleri ekleme, yeni gercek uydurma, dogal Turkce uret. `<think>` bloklari ciktiya sizmamali.

Bu revizyonda hedef quality polish degil; ilk smoke-testte gorulen leakage ve grounding ihlallerini azaltmaktir. Ilk smoke-test pass rate: `64.86%`.

Ek kurallar:
- only final answer
- never output reasoning
- never echo task instructions
- never repeat forbidden examples


In [ ]:
import re
from transformers import AutoTokenizer

THINK_BLOCK_RE = re.compile(r'<think>.*?</think>', re.DOTALL | re.IGNORECASE)
INSTRUCTION_PREFIXES = (
    'gorev:',
    'task:',
    'reasoning:',
    'dusunce:',
    'final answer:',
    'cevap:',
    'yanit:',
    'kullanici mesaji:',
    'canonical response:',
    'canonical cevap:',
    'korunacak terimler:',
    'kacinilacak terimler:',
    'yasak terimler:',
)

def strip_think_blocks(text: str) -> str:
    cleaned = THINK_BLOCK_RE.sub('', text)
    return cleaned.strip()

def clean_final_paraphrase(text: str) -> str:
    cleaned = strip_think_blocks(text)
    cleaned = cleaned.replace('```', ' ').strip()
    lines = []
    for raw_line in cleaned.splitlines():
        line = raw_line.strip().strip('-*').strip()
        if not line:
            continue
        lowered = line.casefold()
        if any(lowered.startswith(prefix) for prefix in INSTRUCTION_PREFIXES):
            continue
        lines.append(line)
    cleaned = ' '.join(lines).strip()
    cleaned = re.sub(r'\s+', ' ', cleaned)
    return cleaned

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def render_messages(record: dict) -> str:
    messages = record['messages']
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return clean_final_paraphrase(text)

preview_text = render_messages(train_records[0])
print(preview_text[:2000])


## H. LoRA/QLoRA Config

Ayarlar bilincli olarak konservatiftir. Adapter-only kayit hedeflenir. Egitimden once degerleri tekrar gozden gecirin.


In [ ]:
from peft import LoraConfig

USE_QLORA = True
PER_DEVICE_TRAIN_BATCH_SIZE = 2
PER_DEVICE_EVAL_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 8
NUM_TRAIN_EPOCHS = 1
LEARNING_RATE = 1e-4
LOGGING_STEPS = 10
SAVE_STEPS = 50

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

print(lora_config)


## I. Training Cell

Asagidaki hucre Colab icinde calistirilabilir, ancak once mutlaka gozden gecirilmelidir. Bu sadece smoke-test egitimidir; production training olarak yorumlanmamalidir.


In [ ]:
from datasets import load_dataset
from peft import get_peft_model
from transformers import (
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
    TrainingArguments,
)
from trl import SFTTrainer

dataset = load_dataset(
    'json',
    data_files={
        'train': str(TRAIN_FILE),
        'validation': str(VALID_FILE),
    },
)

def to_text(example):
    text = tokenizer.apply_chat_template(
        example['messages'],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {'text': strip_think_blocks(text)}

dataset = dataset.map(to_text)

bnb_config = None
if USE_QLORA:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype='float16',
    )

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    quantization_config=bnb_config,
    device_map='auto',
)
model = get_peft_model(model, lora_config)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    eval_strategy='steps',
    eval_steps=SAVE_STEPS,
    bf16=False,
    fp16=True,
    report_to='none',
    save_total_limit=2,
    remove_unused_columns=False,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
    processing_class=tokenizer,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

print('Egitim hucreleri hazir. Ayarlari incelemeden trainer.train() calistirmayin.')
# trainer.train()
# trainer.model.save_pretrained(OUTPUT_DIR)
# tokenizer.save_pretrained(OUTPUT_DIR)


## J. Held-Out Validation Generation

Egitimden sonra held-out validation reference dosyasindaki her `id` icin tek bir paraphrase uretin ve scorer ile degerlendirin. Cikti yalnizca final paraphrase olmali; reasoning, gorev metni veya yasak ornek tekrar etmeyin.


In [ ]:
import json
from pathlib import Path

def build_generation_messages(reference_record: dict) -> list[dict]:
    preserve_terms = ', '.join(reference_record['must_preserve_terms']) or '(yok)'
    forbidden_terms = ', '.join(reference_record['must_not_introduce']) or '(yok)'
    user_prompt = (
        'Yalnizca tek bir final paraphrase yaz.\n'
        'Reasoning, aciklama, gorev metni, etiket, liste veya ornek tekrar etme.\n'
        'Yasakli ornekleri yanitta aynen yazma; bunlar sadece kacinilacak orneklerdir.\n\n'
        f"KULLANICI MESAJI:\n{reference_record.get('user_message', '')}\n\n"
        f"CANONICAL RESPONSE:\n{reference_record['canonical_response']}\n\n"
        f"KORUNMASI ZORUNLU TERIMLER:\n{preserve_terms}\n\n"
        f"KESINLIKLE EKLEME / TEKRAR ETME:\n{forbidden_terms}\n\n"
        'KURALLAR:\n'
        '- yeni bilgi uydurma\n'
        '- korunacak terimlerin hepsini final paraphrase icinde koru\n'
        '- yasakli terimleri kullanma veya ima etme\n'
        '- sadece dogal Turkce final paraphrase ver\n'
    )
    return [
        {
            'role': 'system',
            'content': 'Sen grounded Turkish waiter paraphraser modelisin. Yalnizca final paraphrase ver. Asla reasoning yazma, gorev metnini tekrar etme, <think> etiketi yazma, yasakli ornekleri kopyalama. Yeni menu bilgisi, fiyat, alerji iddiasi veya siparis gercegi ekleme.',
        },
        {'role': 'user', 'content': user_prompt},
    ]

valid_reference_records = load_jsonl(VALID_REFERENCE_FILE)
generation_results = []

for record in valid_reference_records:
    prompt_text = tokenizer.apply_chat_template(
        build_generation_messages(record),
        tokenize=False,
        add_generation_prompt=True,
    )
    prompt_inputs = tokenizer(prompt_text, return_tensors='pt').to(model.device)
    output_tokens = model.generate(
        **prompt_inputs,
        max_new_tokens=128,
        do_sample=False,
        temperature=0.0,
    )
    generated_text = tokenizer.decode(output_tokens[0][prompt_inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    generated_text = clean_final_paraphrase(generated_text)

    generation_results.append({
        'id': record['id'],
        'generated_paraphrase': generated_text,
        'backend_name': 'qwen3_0_6b_grounded_paraphraser_smoke',
        'metadata': {
            'intent': record.get('intent'),
            'reference_type': 'grounded_paraphrase_valid_reference',
            'model_name': MODEL_NAME,
        },
    })

with Path(GENERATED_OUTPUTS_PATH).open('w', encoding='utf-8') as handle:
    for row in generation_results:
        handle.write(json.dumps(row, ensure_ascii=False) + '\n')

print('Yazildi:', GENERATED_OUTPUTS_PATH)
print('Toplam valid cikti:', len(generation_results))


## K. Export / Download Section

Adapter klasorunu ve generated JSONL dosyasini disa aktarip bu repoya geri tasiyin. JSONL icin hedef yol `robot_waiter_ai/evals/generated_grounded_paraphrase_qwen3_0_6b_smoke.jsonl` olmalidir. Sonra yerelde tek scorer komutuyla degerlendirin.


In [ ]:
from pathlib import Path
from shutil import make_archive

archive_base = '/content/grounded_paraphraser_qwen3_0_6b_adapter'
if Path(OUTPUT_DIR).exists():
    archive_path = make_archive(archive_base, 'zip', OUTPUT_DIR)
    print('Adapter arsivi:', archive_path)
else:
    print('Adapter klasoru henuz yok. Egitim kaydi almadiysaniz bu normaldir.')

print('Generated outputs:', GENERATED_OUTPUTS_PATH)
print('Repo hedef yolu: robot_waiter_ai/evals/generated_grounded_paraphrase_qwen3_0_6b_smoke.jsonl')
print('Yerel scoring komutu:')
print(r'.venv\Scripts\python.exe -m robot_waiter_ai.evals.grounded_paraphrase_output_scorer --reference robot_waiter_ai/evals/grounded_paraphrase_valid_reference.jsonl --outputs robot_waiter_ai/evals/generated_grounded_paraphrase_qwen3_0_6b_smoke.jsonl')
